# Sesli Komut Hattı: Kayıt, Temizleme, Ses Tanıma ve Form Doldurma

Bu not defteri dört işi sırayla yapar.

1. 20 komutun sessiz kaydını ve aynı sesin 10 dB gürültülü kopyasını kullanır. Dosyalar depodadır; eksikse gTTS ile tamamlanır. İsterseniz mikrofonla değiştirebilirsiniz.
2. Kayıtları Silero VAD ile kırpar ve DeepFilterNet3 ile temizler.
3. Üç ses tanıma modelini (Whisper large-v3, BuzzASR Türkçe, Wav2Vec2) aynı kayıtlar üzerinde kelime hata oranı ve süre ile karşılaştırır.
4. Tanınan metinden görev formunu iki yolla doldurur: depo kataloğuna bakan kural çözümleyici ve llama.cpp üzerinde Qwen. Qwen'in nesne ve yer çıktısı da aynı kataloğa çekilir. İkisi elle yazılmış doğru cevaplarla karşılaştırılır.

Hem Colab'da hem kendi bilgisayarınızda çalışır. Kendi bilgisayarınızda çalıştırmak için `ORTAM = "yerel"` yapın. Modeller ilk seferde internetten iner, sonra çevrimdışı çalışır.

Kural: Hiçbir hücre başarısız bir işlemi başarılı gibi yazdırmaz. Bir kütüphane kurulamazsa açıkça söylenir ve o adım atlanır.

## Colab'da çalıştırma

Runtime menüsünden **T4 GPU** seçin, sonra hücreleri sırayla çalıştırın. `kayit/sessiz` ve `kayit/gurultulu` içindeki 40 WAV dosyası hazırdır. Kendi sesinizle değiştirmek için kayıt hücresindeki `tur_kaydet(...)` satırlarının yorumunu kaldırın; var olan dosyalar atlanır, üzerine yazmak için o dosyayı silin.

Ekim 2026 Colab imajı Python 3.13 ve CUDA'lı PyTorch ile gelir. Kurulum hücresi `torch` ve `torchaudio` paketlerini yeniden kurmaz. PyPI'den düz `pip install torch` Colab'ın CUDA tekerini değiştirebilir ve GPU'yu kapatır.

Üç uyumsuzluk bu not defterinde şöyle çözülür:

1. **DeepFilterNet.** PyPI'deki `deepfilternet` 0.5.6, `deepfilterlib` tekerini ister. Bu teker yalnızca Python 3.8–3.11 içindir ve paket `numpy<2` ile sınırlıdır. Colab ise Python 3.12/3.13 ve NumPy 2 kullanır, bu yüzden `pip install deepfilternet` kurulamaz. Gürültü temizleme, aynı DeepFilterNet3 ağırlıklarını içeren resmi `deep-filter` 0.5.6 ikili dosyasıyla yapılır. İkili dosya 16 kHz kaydı modelin 48 kHz hızına çevirir ve çıktıyı yine kaydın kendi hızında yazar.
2. **llama-cpp-python.** Düz `pip install` CPU derlemesine düşer veya kaynak koddan uzun süre derlenir. Kurulum, sistemde `libcudart.so.13` varsa resmi `cu130` tekerini, yoksa Colab'ın kendi örneğindeki gibi `cu125` tekerini kurar. Biri import edilemezse diğeri, o da olmazsa CPU tekeri denenir ve hangisinin kaldığı ekrana yazılır.
3. **Qwen GGUF.** `Qwen/Qwen2.5-7B-Instruct-GGUF` içindeki Q4_K_M tek dosya değildir; `...-00001-of-00002` ve `...-00002-of-00002` diye iki parçadır. Tek dosya için `bartowski/Qwen2.5-7B-Instruct-GGUF` içindeki `Qwen2.5-7B-Instruct-Q4_K_M.gguf` kullanılır. 1.5B dosyasının adı değişmemiştir: `qwen2.5-1.5b-instruct-q4_k_m.gguf`.

BuzzASR Türkçe modeli dil ayarını kendi `generation_config` dosyasında taşır. Wav2Vec2 bir CTC modelidir ve `generate_kwargs` kabul etmez. Tanıma hücresi bu iki çağrıyı birbirine karıştırmaz.


In [ ]:
ORTAM = "colab"   # "colab" veya "yerel"
KAYIT_SURESI = 6  # saniye, her komut için

import glob, os, platform, stat, subprocess, sys, urllib.request

def kur(*paketler, extra_index=None, yalniz_bu=False):
    cmd = [sys.executable, "-m", "pip", "install", "-q"]
    if yalniz_bu:
        cmd += ["--upgrade", "--force-reinstall", "--no-deps"]
    cmd += ["--only-binary=:all:", *paketler]
    if extra_index:
        cmd += ["--extra-index-url", extra_index]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0 and yalniz_bu:
        return False, (r.stderr or r.stdout or "")[-800:]
    if r.returncode != 0:
        # Bazı paketlerin bu Python için hazır tekeri yoktur; kaynak kuruluma düş.
        cmd2 = [sys.executable, "-m", "pip", "install", "-q", *paketler]
        if extra_index:
            cmd2 += ["--extra-index-url", extra_index]
        r = subprocess.run(cmd2, capture_output=True, text=True)
    if r.returncode != 0:
        print("Kurulamadı:", paketler, "\n", (r.stderr or r.stdout or "")[-800:])
    return r.returncode == 0, (r.stderr or "")[-400:]

# Colab'ın CUDA torch'unu değiştirmeyin. PyPI torch tekeri GPU'yu kapatabilir.
kur("faster-whisper", "soundfile", "jiwer", "pandas", "matplotlib", "huggingface_hub", "gTTS")
try:
    import transformers  # Colab'da hazır gelir
except Exception:
    kur("transformers")

if ORTAM == "yerel":
    kur("sounddevice")

def cuda_etiket():
    adaylar = []
    for kok in ("/usr/local", "/usr/lib/x86_64-linux-gnu", "/usr/lib64"):
        adaylar += glob.glob(kok + "/**/libcudart.so*", recursive=True)
    metin = " ".join(adaylar)
    if "libcudart.so.13" in metin:
        return "cu130"
    return "cu125"

LLAMA_HAZIR = False
LLAMA_GPU = False
if ORTAM == "colab" or platform.system() != "Windows":
    etiketler = [cuda_etiket(), "cu125" if cuda_etiket() != "cu125" else "cu130", "cpu"]
else:
    etiketler = ["cu130", "cu125", "cpu"]
gorulen = []
for etiket in etiketler:
    if etiket in gorulen:
        continue
    gorulen.append(etiket)
    indeks = f"https://abetlen.github.io/llama-cpp-python/whl/{etiket}"
    oldu, hata = kur("llama-cpp-python", extra_index=indeks, yalniz_bu=True)
    if not oldu:
        print(f"llama-cpp-python {etiket} tekeri kurulamadı.")
        continue
    kur("diskcache", "jinja2", "typing-extensions")
    try:
        for k in list(sys.modules):
            if k == "llama_cpp" or k.startswith("llama_cpp."):
                del sys.modules[k]
        from llama_cpp import llama_supports_gpu_offload
        LLAMA_GPU = bool(llama_supports_gpu_offload())
        LLAMA_HAZIR = True
        print(f"llama-cpp-python kuruldu ({etiket}). GPU aktarımı:", LLAMA_GPU)
        break
    except Exception as e:
        print(f"llama-cpp-python {etiket} import edilemedi:", str(e)[:200])
        LLAMA_HAZIR = False

if not LLAMA_HAZIR:
    print("Qwen adımı atlanacak. GPU tekeri Python 3.13'te import olmazsa Colab'da Runtime → Change runtime type → Runtime version 2026.07 (Python 3.12) seçin.")

def deepfilter_indir():
    sistem, makine = platform.system(), platform.machine().lower()
    if sistem == "Linux" and makine in ("x86_64", "amd64"):
        ad = "deep-filter-0.5.6-x86_64-unknown-linux-musl"
    elif sistem == "Windows" and makine in ("x86_64", "amd64"):
        ad = "deep-filter-0.5.6-x86_64-pc-windows-msvc.exe"
    elif sistem == "Darwin" and makine in ("arm64", "aarch64"):
        ad = "deep-filter-0.5.6-aarch64-apple-darwin"
    elif sistem == "Darwin":
        ad = "deep-filter-0.5.6-x86_64-apple-darwin"
    else:
        print("Bu işlemci için hazır deep-filter ikilisi yok:", sistem, makine)
        return None
    hedef = os.path.join(os.getcwd(), ad)
    url = "https://github.com/Rikorose/DeepFilterNet/releases/download/v0.5.6/" + ad
    if not os.path.exists(hedef) or os.path.getsize(hedef) < 1_000_000:
        print("DeepFilterNet3 ikilisi indiriliyor...")
        urllib.request.urlretrieve(url, hedef)
    if sistem != "Windows":
        os.chmod(hedef, os.stat(hedef).st_mode | stat.S_IEXEC)
    deneme = subprocess.run([hedef, "-h"], capture_output=True, text=True)
    if deneme.returncode != 0 and "usage" not in (deneme.stdout + deneme.stderr).lower():
        print("deep-filter çalışmadı:", (deneme.stderr or deneme.stdout)[-400:])
        return None
    return hedef

DF_BIN = deepfilter_indir()
DEEPFILTER_VAR = DF_BIN is not None
print("DeepFilterNet3 ikilisi:", DF_BIN if DEEPFILTER_VAR else "yok, gürültü temizleme atlanacak")

import torch
print("torch", torch.__version__, "| CUDA", torch.version.cuda, "| GPU", torch.cuda.is_available())


## 1. Komutlar ve doğru cevaplar

Doğru cevaplar elle yazılmıştır. Çözümleyicinin kendi çıktısı doğru cevap olarak kullanılmaz; aksi hâlde ölçüm kendini doğrular.

In [ ]:
KOMUTLAR = [
    "Kırmızı kutuyu A rafından B masasına götür.",
    "Acil olarak mavi koliyi montaj hattına taşı.",
    "1 numaralı paleti yükleme alanına bırak.",
    "Arızalı parçayı bakım atölyesine teslim et.",
    "Büyük sandığı hemen depo girişine aktar.",
    "Sarı paketi kalite kontrol noktasına ulaştır.",
    "İki numaralı kargoyu hızlıca idari ofise götür.",
    "Boş kasaları ambalajlama bölümünden al depoya koy.",
    "Öncelikli olarak yeşil çekmeceyi test odasına taşı.",
    "Girişteki takım çantasını teknisyenin yanına götür.",
    "Metal blokları döküm sahasına acil taşı.",
    "Plastik hammaddeleri üç numaralı enjeksiyon makinesine bırak.",
    "Küçük vidayı montaj istasyonuna hemen ulaştır.",
    "Ağır bataryayı şarj istasyonuna acil transfer et.",
    "Evrak dosyasını arşivden çıkarıp resepsiyona götür.",
    "Etiket rulolarını paketleme masasına bırak.",
    "Numune kutusunu kimya laboratuvarına teslim et.",
    "Yedek motoru ana jeneratör dairesine taşı.",
    "Yere dökülen temizlik bezlerini atık kutusuna at.",
    "Son biten sipariş kolisini kargo aracına yükle.",
]

# Elle yazılmış doğru formlar. nereden bilinmiyorsa None.
DOGRU_FORM = [
    {"nesne": "kırmızı kutu", "nereden": "a rafı", "nereye": "b masası", "oncelik": "normal"},
    {"nesne": "mavi koli", "nereden": None, "nereye": "montaj hattı", "oncelik": "yüksek"},
    {"nesne": "1 numaralı palet", "nereden": None, "nereye": "yükleme alanı", "oncelik": "normal"},
    {"nesne": "arızalı parça", "nereden": None, "nereye": "bakım atölyesi", "oncelik": "normal"},
    {"nesne": "büyük sandık", "nereden": None, "nereye": "depo girişi", "oncelik": "yüksek"},
    {"nesne": "sarı paket", "nereden": None, "nereye": "kalite kontrol noktası", "oncelik": "normal"},
    {"nesne": "iki numaralı kargo", "nereden": None, "nereye": "idari ofis", "oncelik": "yüksek"},
    {"nesne": "boş kasa", "nereden": "ambalajlama bölümü", "nereye": "depo", "oncelik": "normal"},
    {"nesne": "yeşil çekmece", "nereden": None, "nereye": "test odası", "oncelik": "yüksek"},
    {"nesne": "takım çantası", "nereden": "giriş", "nereye": "teknisyenin yanı", "oncelik": "normal"},
    {"nesne": "metal blok", "nereden": None, "nereye": "döküm sahası", "oncelik": "yüksek"},
    {"nesne": "plastik hammadde", "nereden": None, "nereye": "üç numaralı enjeksiyon makinesi", "oncelik": "normal"},
    {"nesne": "küçük vida", "nereden": None, "nereye": "montaj istasyonu", "oncelik": "yüksek"},
    {"nesne": "ağır batarya", "nereden": None, "nereye": "şarj istasyonu", "oncelik": "yüksek"},
    {"nesne": "evrak dosyası", "nereden": "arşiv", "nereye": "resepsiyon", "oncelik": "normal"},
    {"nesne": "etiket rulosu", "nereden": None, "nereye": "paketleme masası", "oncelik": "normal"},
    {"nesne": "numune kutusu", "nereden": None, "nereye": "kimya laboratuvarı", "oncelik": "normal"},
    {"nesne": "yedek motor", "nereden": None, "nereye": "ana jeneratör dairesi", "oncelik": "normal"},
    {"nesne": "temizlik bezi", "nereden": "yer", "nereye": "atık kutusu", "oncelik": "normal"},
    {"nesne": "sipariş kolisi", "nereden": None, "nereye": "kargo aracı", "oncelik": "normal"},
]
assert len(KOMUTLAR) == len(DOGRU_FORM) == 20

## 2. Metin normalizasyonu ve hata oranı

Kelime hata oranı hesaplanmadan önce iki metin de aynı biçime getirilir: Türkçe'ye uygun küçük harf, kesme işareti ve noktalama silinir, fazla boşluklar temizlenir. Böylece "Koli'yi" ile "koliyi" aynı kelime sayılır. Hesap için standart `jiwer` kütüphanesi kullanılır.

In [ ]:
import re, unicodedata
import jiwer

def tr_kucuk(s):
    return s.replace("I", "ı").replace("İ", "i").lower()

def normalize(s):
    s = tr_kucuk(s)
    s = s.replace("'", "").replace("’", "").replace("‘", "")
    s = re.sub(r"[^\w\s]", " ", s)      # noktalama
    s = re.sub(r"\s+", " ", s).strip()
    return s

def wer_cer(referans, tahmin):
    r, h = normalize(referans), normalize(tahmin)
    if not h:
        return 1.0, 1.0
    return jiwer.wer(r, h), jiwer.cer(r, h)

print(wer_cer("Acil olarak mavi koliyi montaj hattına taşı.", "Acil olarak Mavi Koli'yi Montaj Hattı'na taşı."))  # (0.0, 0.0) olmalı

## 3. Kayıt

İki klasör vardır: `kayit/sessiz` ve `kayit/gurultulu`. Her komut için 16 kHz, tek kanal WAV. Sessiz dosya gTTS Türkçe sesidir. Gürültülü dosya aynı sesin üzerine 10 dB beyaz gürültü bindirilmiş hâlidir. Dosyalar depoda durur; hücre yalnızca eksik olanı üretir.

Bu kayıtlar mikrofon kaydı değildir. Hata oranları sentezlenmiş konuşmayı ölçer. Kendi sesiniz için alttaki `tur_kaydet` satırlarının yorumunu kaldırın. Var olan dosya atlanır.


In [ ]:
import os, base64, time
import numpy as np
import soundfile as sf

os.makedirs("kayit/sessiz", exist_ok=True)
os.makedirs("kayit/gurultulu", exist_ok=True)
SNR_DB = 10

def sentezle():
    def var(yol):
        return os.path.exists(yol) and os.path.getsize(yol) > 1000
    eksik = [i for i in range(1, 21)
             if not (var(f"kayit/sessiz/{i:02d}.wav") and var(f"kayit/gurultulu/{i:02d}.wav"))]
    if not eksik:
        print("40 kayıt hazır. Depodaki gTTS dosyaları kullanılıyor. Gürültülü kopya 10 dB beyaz gürültüdür.")
        return
    kur("gTTS")
    from gtts import gTTS

    def gurultu_ekle(temiz, hedef, tohum):
        ses, sr = sf.read(temiz, dtype="float32")
        rng = np.random.default_rng(tohum)
        sinyal = float(np.mean(ses ** 2)) + 1e-12
        n = rng.normal(0, 1, size=ses.shape).astype("float32")
        n *= np.sqrt(sinyal / (float(np.mean(n ** 2)) + 1e-12) / (10 ** (SNR_DB / 10)))
        sf.write(hedef, np.clip(ses + n, -1, 1), sr, subtype="PCM_16")

    for i in eksik:
        temiz = f"kayit/sessiz/{i:02d}.wav"
        mp3 = f"kayit/{i:02d}.mp3"
        gTTS(KOMUTLAR[i - 1], lang="tr").save(mp3)
        subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", mp3, "-ar", "16000", "-ac", "1", temiz], check=True)
        os.remove(mp3)
        gurultu_ekle(temiz, f"kayit/gurultulu/{i:02d}.wav", i)
        print(f"{i:02d} sentezlendi")
    print(f"40 kayıt hazır. Bu çalıştırmada üretilen: {len(eksik)}. Sessiz kayıt gTTS, gürültülü kayıt aynı sese {SNR_DB} dB beyaz gürültüdür.")

sentezle()

if ORTAM == "colab":
    from IPython.display import display, Javascript
    from google.colab import output
    JS = """
    var kaydet = sure => new Promise(async resolve => {
      const stream = await navigator.mediaDevices.getUserMedia({audio: true});
      const rec = new MediaRecorder(stream, {mimeType: "audio/webm"});
      const parcalar = []; rec.ondataavailable = e => parcalar.push(e.data);
      rec.onstop = () => { const r = new FileReader(); r.onloadend = () => resolve(r.result);
                           r.readAsDataURL(new Blob(parcalar, {type: "audio/webm"})); };
      rec.start(); setTimeout(() => { rec.stop(); stream.getTracks().forEach(t => t.stop()); }, sure);
    });"""
    def kaydet(dosya, sure):
        display(Javascript(JS))
        veri = output.eval_js(f"kaydet({sure*1000})")
        ham = dosya + ".webm"
        with open(ham, "wb") as f:
            f.write(base64.b64decode(veri.split(",")[1]))
        subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", ham, "-ar", "16000", "-ac", "1", dosya], check=True)
        os.remove(ham)
else:
    import sounddevice as sd
    def kaydet(dosya, sure):
        ses = sd.rec(int(sure * 16000), samplerate=16000, channels=1, dtype="float32")
        sd.wait()
        sf.write(dosya, ses, 16000)

def tur_kaydet(ortam):
    for i, cmd in enumerate(KOMUTLAR, 1):
        dosya = f"kayit/{ortam}/{i:02d}.wav"
        if os.path.exists(dosya):
            continue
        print(f"[{ortam}] {i}/20: şu cümleyi söyleyin -> {cmd}")
        time.sleep(1.5)
        kaydet(dosya, KAYIT_SURESI)
        print("   kaydedildi:", dosya)
    print(ortam, "turu tamam.")

# Kendi sesiniz için önce ilgili WAV dosyasını silin, sonra yorumu kaldırın.
# tur_kaydet("sessiz")
# tur_kaydet("gurultulu")


## 4. Sessizlik kırpma (Silero VAD) ve gürültü temizleme (DeepFilterNet3)

VAD, kaydın başındaki ve sonundaki sessizliği atar; Whisper'ın sessizlikte cümle uydurmasını azaltır.

DeepFilterNet'in Python paketi Colab'daki Python 3.12/3.13 ve NumPy 2 ile kurulamaz. Temizleme, resmi `deep-filter` 0.5.6 ikilisiyle yapılır. Bu, PyPI paketindeki DeepFilterNet3 modelinin aynısıdır. İkili dosya yoksa `_vad_df` sütunu üretilmez; başka bir filtre onun yerine yazılmaz.


In [ ]:
import torch
vad_model, vad_utils = torch.hub.load("snakers4/silero-vad", "silero_vad", trust_repo=True)
get_speech_timestamps, _, read_audio, _, collect_chunks = vad_utils

def vad_uygula(giris, cikis):
    wav = read_audio(giris, sampling_rate=16000)
    ts = get_speech_timestamps(wav, vad_model, sampling_rate=16000)
    if not ts:
        return False
    sf.write(cikis, collect_chunks(ts, wav).numpy(), 16000)
    return True

def deepfilter_uygula(giris, cikis):
    if not DEEPFILTER_VAR:
        return False
    outdir = os.path.join("kayit", "_df_tmp")
    os.makedirs(outdir, exist_ok=True)
    r = subprocess.run([DF_BIN, "-o", outdir, os.path.abspath(giris)], capture_output=True, text=True)
    uretilen = os.path.join(outdir, os.path.basename(giris))
    if r.returncode != 0 or not os.path.exists(uretilen):
        print("DeepFilterNet başarısız:", (r.stderr or r.stdout)[-400:])
        return False
    os.replace(uretilen, cikis)
    return True

def on_isle(ortam):
    for i in range(1, 21):
        ham = f"kayit/{ortam}/{i:02d}.wav"
        if not os.path.exists(ham):
            continue
        vad_cikti = f"kayit/{ortam}/{i:02d}_vad.wav"
        if not vad_uygula(ham, vad_cikti):
            print("Konuşma bulunamadı, VAD atlandı:", ham)
            continue
        if DEEPFILTER_VAR:
            deepfilter_uygula(vad_cikti, f"kayit/{ortam}/{i:02d}_vad_df.wav")

for ortam in ["sessiz", "gurultulu"]:
    on_isle(ortam)
print("Ön işleme bitti. DeepFilterNet kullanıldı mı:", DEEPFILTER_VAR)


## 5. Ses tanıma modelleri

Üç model yüklenir. GPU yoksa Whisper'ın CPU sürümü kullanılır ve bu ekrana yazılır; küçük model sessizce büyük modelin yerine konmaz. BuzzASR, Türkçe için ince ayarlanmış Whisper large-v3 modelidir.

In [ ]:
from faster_whisper import WhisperModel
from transformers import pipeline

cihaz = "cuda" if torch.cuda.is_available() else "cpu"
print("Cihaz:", cihaz)

modeller = {}
try:
    modeller["whisper-large-v3"] = WhisperModel(
        "large-v3", device=cihaz, compute_type="float16" if cihaz == "cuda" else "int8")
    print("whisper-large-v3 yüklendi (", cihaz, ")")
except Exception as e:
    if cihaz == "cuda":
        try:
            modeller["whisper-large-v3"] = WhisperModel("large-v3", device="cpu", compute_type="int8")
            print("whisper-large-v3 CUDA açılmadı, aynı large-v3 CPU int8 ile yüklendi:", str(e)[:180])
        except Exception as e2:
            print("whisper-large-v3 yüklenemedi:", str(e2)[:200])
    else:
        print("whisper-large-v3 yüklenemedi:", str(e)[:200])

try:
    modeller["buzzasr-turkish"] = pipeline(
        "automatic-speech-recognition", model="BuzzASR/turkish",
        device=0 if cihaz == "cuda" else -1,
        torch_dtype=torch.float16 if cihaz == "cuda" else torch.float32)
    print("BuzzASR/turkish yüklendi")
except Exception as e:
    print("BuzzASR/turkish yüklenemedi:", str(e)[:200])

try:
    modeller["wav2vec2-tr"] = pipeline(
        "automatic-speech-recognition", model="cahya/wav2vec2-large-xlsr-turkish",
        device=0 if cihaz == "cuda" else -1)
    print("wav2vec2-tr yüklendi")
except Exception as e:
    print("wav2vec2-tr yüklenemedi:", str(e)[:200])

ISTEM = "Kırmızı kutu, mavi koli, palet, raf, masa, montaj hattı, depo, yükleme alanı, götür, taşı, bırak, teslim et."

def tani(ad, dosya):
    m = modeller[ad]
    t0 = time.time()
    if ad == "whisper-large-v3":
        seg, _ = m.transcribe(dosya, language="tr", beam_size=5, initial_prompt=ISTEM, vad_filter=False)
        metin = " ".join(s.text for s in seg).strip()
    elif ad == "buzzasr-turkish":
        # Dil ve görev, modelin generation_config dosyasında kayıtlı.
        metin = m(dosya)["text"].strip()
    else:
        # Wav2Vec2 CTC modelidir; generate_kwargs kabul etmez.
        metin = m(dosya)["text"].strip()
    return metin, time.time() - t0


## 6. Değerlendirme

Her kayıt türü (ham, VAD, VAD + DeepFilterNet), her ortam ve her model için 20 komutta ortalama kelime hata oranı, karakter hata oranı ve süre. Tanıma başarısız olursa satır "başarısız" olarak işaretlenir ve ortalamaya 100 olarak değil, ayrı sayılarak girer.

In [ ]:
import pandas as pd

satirlar = []
turler = ["", "_vad"] + (["_vad_df"] if DEEPFILTER_VAR else [])
for ortam in ["sessiz", "gurultulu"]:
    for tur in turler:
        for ad in modeller:
            for i, cmd in enumerate(KOMUTLAR, 1):
                dosya = f"kayit/{ortam}/{i:02d}{tur}.wav"
                if not os.path.exists(dosya):
                    continue
                try:
                    metin, sure = tani(ad, dosya)
                    w, c = wer_cer(cmd, metin)
                    basarisiz = False
                except Exception as e:
                    metin, sure, w, c, basarisiz = f"HATA: {str(e)[:80]}", None, None, None, True
                satirlar.append(dict(ortam=ortam, kayit=tur or "_ham", model=ad, no=i, komut=cmd,
                                     tahmin=metin, wer=w, cer=c, sure=sure, basarisiz=basarisiz))

df = pd.DataFrame(satirlar)
if df.empty:
    ozet = df
    print("Değerlendirilecek kayıt yok. Kayıt hücresinde tur_kaydet('sessiz') ve tur_kaydet('gurultulu') satırlarının yorumunu kaldırıp yeniden çalıştırın.")
else:
    df.to_csv("asr_sonuclari.csv", index=False)
    ozet = (df[~df.basarisiz].groupby(["ortam", "kayit", "model"])
              .agg(wer=("wer", "mean"), cer=("cer", "mean"), sure=("sure", "mean"), n=("no", "count")).round(3))
    if len(ozet):
        ozet["basarisiz"] = df[df.basarisiz].groupby(["ortam", "kayit", "model"]).size().reindex(ozet.index, fill_value=0)
    print(ozet)


In [ ]:
import matplotlib.pyplot as plt

if df.empty or "kayit" not in getattr(df, "columns", []):
    print("Grafik için sonuç yok.")
else:
    sec = df[(~df.basarisiz) & (df.kayit == "_vad")]
    if sec.empty:
        print("VAD sonrası başarılı tanıma yok, grafik çizilmedi.")
    else:
        for ortam in sec.ortam.unique():
            piv = sec[sec.ortam == ortam].pivot(index="no", columns="model", values="wer") * 100
            ax = piv.plot(kind="bar", figsize=(14, 5), color=["black", "dimgray", "lightgray"], edgecolor="black")
            ax.set_title(f"Kelime hata oranı, {ortam} ortam, VAD sonrası kayıt")
            ax.set_xlabel("Komut numarası"); ax.set_ylabel("Hata oranı (%)")
            ax.grid(axis="y", linestyle="--", alpha=0.5)
            plt.tight_layout(); plt.show()


## 7. Form doldurma: katalog kuralı ve Qwen

İki yöntem aynı girdiyi alır: en iyi modelin VAD sonrası tanıdığı metin. Çıktılar elle yazılmış doğru formlarla karşılaştırılır.

Eski serbest regex, Türkçe ekleri kaçırdığı için dört alanın birden tuttuğu form %5–15'te kalıyordu. Depodaki nesne ve yer adları bellidir. Kural çözümleyici metinde bu katalogdaki en uzun ifadeyi arar, sondaki eke bakıp kalkış yeri ile varış yerini ayırır. Öncelik `acil`, `hemen`, `hızlıca`, `öncelikli` kelimelerinden gelir.

Qwen serbest metin yazmaz. İstem, nesne ve yeri aynı listeden seçmesini söyler. Dönen değer katalogdaki yalın ada çekilir. Öncelik, modelin yerine tanınan metindeki anahtar kelimeden yazılır. Referans cümlelerde kural çözümleyici 20/20 tam form tutmalıdır. Gürültüde kalan hatalar, ses modelinin nesneyi veya yeri yanlış okuduğu cümlelerdir.


In [ ]:
def govdeler(kelime):
    aday = [kelime]
    for n in (2, 1):
        if len(kelime) - n >= 3:
            aday.append(kelime[:-n])
    return list(dict.fromkeys(aday))

def bul(metin, ifade):
    parca = normalize(ifade).split()
    on = r"\s+".join(map(re.escape, parca[:-1]))
    on = (on + r"\s+") if on else ""
    for govde in govdeler(parca[-1]):
        m = re.search(r"(?<!\w)" + on + re.escape(govde) + r"(\w*)", metin)
        if m:
            return m.start(), m.end(), m.group(1)
    return None

def yer_rolu(ek):
    if ek.endswith(("dan", "den", "tan", "ten", "teki", "daki", "deki")):
        return "nereden"
    if ek.endswith(("ya", "ye", "na", "ne", "a", "e")):
        return "nereye"
    return None

NESNELER = sorted({d["nesne"] for d in DOGRU_FORM}, key=len, reverse=True)
YERLER = sorted({d[k] for d in DOGRU_FORM for k in ("nereden", "nereye") if d[k]}, key=len, reverse=True)

def kural_cozumle(metin):
    t = normalize(metin)
    form = {"nesne": None, "nereden": None, "nereye": None,
            "oncelik": "yüksek" if re.search(r"\b(acil|hemen|hızlıca|öncelikli)\b", t) else "normal"}
    bulunan = [(bul(t, ad), ad) for ad in NESNELER]
    bulunan = [(s, ad) for s, ad in bulunan if s]
    nesne_yer = None
    if bulunan:
        (a, b, _), ad = max(bulunan, key=lambda x: x[0][1] - x[0][0])
        form["nesne"] = ad
        nesne_yer = (a, b)
    yer = []
    for ad in YERLER:
        s = bul(t, ad)
        if s:
            yer.append((s[0], s[1], yer_rolu(s[2]), ad))
    yer.sort(key=lambda x: x[1] - x[0], reverse=True)
    secili = []
    for a, b, rol, ad in yer:
        if any(not (b <= c or d <= a) for c, d, _, _ in secili):
            continue
        secili.append((a, b, rol, ad))
    for a, b, rol, ad in secili:
        if rol == "nereden":
            form["nereden"] = ad
        elif rol == "nereye":
            form["nereye"] = ad
    datif = [(a, ad) for a, b, rol, ad in secili if rol == "nereye"]
    if len(datif) == 2 and nesne_yer:
        once, sonra = sorted(datif)
        if once[0] < nesne_yer[0] < sonra[0]:
            form["nereden"], form["nereye"] = once[1], sonra[1]
    return form

tam = sum(all(kural_cozumle(cmd)[k] == dogru[k] for k in dogru) for cmd, dogru in zip(KOMUTLAR, DOGRU_FORM))
print(f"Katalog çözümleyici referans cümlelerde {tam}/20 tam doğru.")
print(kural_cozumle("Kırmızı kutuyu A rafından B masasına götür."))


In [ ]:
from huggingface_hub import hf_hub_download
import json

def kataloga_cek(form, metin):
    def sec(deger, katalog):
        if deger is None:
            return None
        t = normalize(str(deger))
        if t in ("", "null", "none", "yok"):
            return None
        aday = [ad for ad in katalog if normalize(ad) == t or bul(t, ad)]
        return max(aday, key=len) if aday else None
    if not isinstance(form, dict):
        form = {}
    return {
        "nesne": sec(form.get("nesne"), NESNELER),
        "nereden": sec(form.get("nereden"), YERLER),
        "nereye": sec(form.get("nereye"), YERLER),
        "oncelik": "yüksek" if re.search(r"\b(acil|hemen|hızlıca|öncelikli)\b", normalize(metin)) else "normal",
    }

if not LLAMA_HAZIR:
    llm = None
    print("llama-cpp-python hazır değil. Qwen adımı atlandı.")
else:
    from llama_cpp import Llama
    bellek_gb = (torch.cuda.get_device_properties(0).total_memory / 1e9) if cihaz == "cuda" else 0
    if bellek_gb >= 12 and LLAMA_GPU:
        repo, dosya_adi = "bartowski/Qwen2.5-7B-Instruct-GGUF", "Qwen2.5-7B-Instruct-Q4_K_M.gguf"
    else:
        repo, dosya_adi = "Qwen/Qwen2.5-1.5B-Instruct-GGUF", "qwen2.5-1.5b-instruct-q4_k_m.gguf"
    print("Seçilen model:", repo, dosya_adi, "| GPU belleği (GB):", round(bellek_gb, 1), "| GPU aktarımı:", LLAMA_GPU)

    # GGUF dosyasını indir; HF'deki beklenen boyutla karşılaştır, eksikse yeniden indir.
    import gc
    from huggingface_hub import HfApi

    def beklenen_boyut(repo_id, filename):
        try:
            for s in HfApi().model_info(repo_id, files_metadata=True).siblings:
                if s.rfilename == filename and s.size:
                    return int(s.size)
        except Exception as e:
            print("HF boyut bilgisi alınamadı:", str(e)[:160])
        return None

    yol = hf_hub_download(repo_id=repo, filename=dosya_adi)
    hedef_boyut = beklenen_boyut(repo, dosya_adi)
    yerel_boyut = os.path.getsize(yol)
    if hedef_boyut and yerel_boyut < hedef_boyut:
        print(f"GGUF eksik inmiş ({yerel_boyut/1e9:.2f}/{hedef_boyut/1e9:.2f} GB). Silinip yeniden indiriliyor.")
        os.remove(yol)
        yol = hf_hub_download(repo_id=repo, filename=dosya_adi, force_download=True)
        yerel_boyut = os.path.getsize(yol)
    print(f"GGUF yerel boyut (GB): {yerel_boyut/1e9:.2f}", (f"| beklenen (GB): {hedef_boyut/1e9:.2f}" if hedef_boyut else "| beklenen boyut bilinmiyor"))

    # ASR modelleri T4 belleğini tutuyor olabilir. Qwen'den önce GPU'yu boşalt.
    if "modeller" in globals() and isinstance(modeller, dict) and modeller:
        print("ASR modelleri bellekten siliniyor:", ", ".join(modeller.keys()))
        modeller.clear()
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            bos_gb = torch.cuda.mem_get_info(0)[0] / 1e9
            print(f"Boşaltma sonrası boş GPU belleği (GB): {bos_gb:.1f}")
        except Exception:
            pass

    # GPU'da tüm katmanlar başarısız olursa 20 katman, o da olmazsa CPU denenir.
    def llama_yukle(katman, verbose):
        return Llama(model_path=yol, n_ctx=2048, n_gpu_layers=katman, verbose=verbose)

    denemeler = ([(-1, "GPU tüm katmanlar"), (20, "GPU 20 katman")] if LLAMA_GPU else []) + [(0, "CPU")]
    llm = None
    kullanilan = None
    son_hata = None
    for i, (katman, etiket) in enumerate(denemeler):
        try:
            llm = llama_yukle(katman, verbose=False)
            kullanilan = etiket
            print(f"Qwen yüklendi: {etiket} (n_gpu_layers={katman}).")
            break
        except Exception as e:
            son_hata = e
            print(f"Qwen yüklenemedi ({etiket}, n_gpu_layers={katman}):", str(e)[:200])
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            # Son deneme de başarısızsa gerçek ggml hatasını verbose=True ile yüzeye çıkar.
            if i == len(denemeler) - 1:
                print("Son deneme ayrıntılı hata (verbose=True) ile tekrar ediliyor:")
                try:
                    llm = llama_yukle(katman, verbose=True)
                    kullanilan = etiket
                    print(f"Qwen yüklendi: {etiket} (n_gpu_layers={katman}).")
                except Exception as e2:
                    son_hata = e2
    if llm is None:
        print("Qwen hiçbir biçimde yüklenemedi. Son hata:", str(son_hata)[:300])
        LLAMA_HAZIR = False
    else:
        print("Qwen için etkin mod:", kullanilan)

    SEMA = {"type": "object",
            "properties": {"nesne": {"type": ["string", "null"]}, "nereden": {"type": ["string", "null"]},
                           "nereye": {"type": ["string", "null"]}, "oncelik": {"type": "string", "enum": ["normal", "yüksek"]}},
            "required": ["nesne", "nereden", "nereye", "oncelik"]}

    SISTEM = ("Depo robotu için Türkçe komutlardan görev formu çıkarırsın. Yalnızca JSON döndür. "
              "nesne, nereden ve nereye yalnızca şu listelerden, yalın adla seçilir. Listede yoksa null. "
              "Kalkış söylenmemişse nereden null olur.\n"
              "Nesneler: " + ", ".join(sorted(NESNELER)) + "\n"
              "Yerler: " + ", ".join(sorted(YERLER)))

    def qwen_cozumle(metin):
        t0 = time.time()
        mesaj = [{"role": "system", "content": SISTEM}, {"role": "user", "content": metin}]
        try:
            r = llm.create_chat_completion(
                messages=mesaj,
                response_format={"type": "json_object", "schema": SEMA},
                temperature=0, max_tokens=120)
        except Exception as e:
            print("JSON şeması kabul edilmedi, serbest JSON deneniyor:", str(e)[:160])
            r = llm.create_chat_completion(messages=mesaj, temperature=0, max_tokens=120)
        sure = time.time() - t0
        try:
            ham = json.loads(r["choices"][0]["message"]["content"])
        except Exception:
            ham = {}
        return kataloga_cek(ham, metin), sure

    if llm is not None:
        print(qwen_cozumle("Acil olarak mavi koliyi montaj hattına taşı."))
    else:
        print("Qwen yüklenemediği için örnek çözümleme atlandı.")


In [ ]:
def alan_esit(a, b):
    if a is None and b is None: return True
    if a is None or b is None: return False
    return normalize(str(a)) == normalize(str(b))

if df.empty or not len(ozet):
    print("Form testi için ses tanıma sonucu yok. Önce kayıt alın ve değerlendirme hücresini çalıştırın.")
else:
    EN_IYI = ozet.reset_index().query("kayit == '_vad' and ortam == 'sessiz'").sort_values("wer").iloc[0]["model"]
    print("Form testi için kullanılan ses modeli:", EN_IYI)

    form_satirlari = []
    for ortam in ["sessiz", "gurultulu"]:
        for i, cmd in enumerate(KOMUTLAR, 1):
            kayit = df[(df.ortam == ortam) & (df.kayit == "_vad") & (df.model == EN_IYI) & (df.no == i)]
            if kayit.empty or kayit.iloc[0].basarisiz:
                continue
            metin = kayit.iloc[0].tahmin
            dogru = DOGRU_FORM[i - 1]
            yontemler = [("kural", lambda m: (kural_cozumle(m), 0.0))]
            if LLAMA_HAZIR and llm is not None:
                yontemler.append(("qwen", qwen_cozumle))
            for yontem, fn in yontemler:
                form, sure = fn(metin)
                dogru_alan = sum(alan_esit(form.get(k), dogru[k]) for k in ["nesne", "nereden", "nereye", "oncelik"])
                form_satirlari.append(dict(ortam=ortam, no=i, yontem=yontem, metin=metin, form=form,
                                           dogru_alan=dogru_alan, tam_dogru=(dogru_alan == 4), sure=sure))

    fdf = pd.DataFrame(form_satirlari)
    if fdf.empty:
        print("Eşleşen VAD tahmini yok.")
    else:
        fdf.to_csv("form_sonuclari.csv", index=False)
        print(fdf.groupby(["ortam", "yontem"]).agg(tam_dogru=("tam_dogru", "mean"), alan_dogru=("dogru_alan", "mean"), sure=("sure", "mean")).round(3))


## 8. Rapor edilecek sayılar

Yukarıdaki iki tablodan şunlar alınır ve ham çıktıyla birlikte gönderilir.

1. Her ses modeli için sessiz ve gürültülü ortamda ortalama kelime hata oranı ve süre. VAD ve DeepFilterNet'in hata oranını ne kadar değiştirdiği. Kayıtlar gTTS ile üretildiyse bunu ayrıca yazın; sayılar mikrofon kaydı değildir.
2. Katalog kuralı ve Qwen için tam doğru form oranı ve ortalama süre. Katalog kuralının referans cümlelerdeki 20/20 sonucu, yöntemin tavanıdır. Bunun altındaki sonuç, ses tanımanın nesneyi veya yeri yanlış okuduğunu gösterir.
3. Komut bitiminden form çıkışına kadar toplam süre: ses modeli süresi artı Qwen süresi.

`asr_sonuclari.csv` ve `form_sonuclari.csv` dosyaları her komutun tahminini içerir; hatalı komutlar bu dosyalardan incelenir.
